# Deep Research Tool — Web UI（ブラウザGUI）利用ガイド

ブラウザから調査を実行できるWeb UIの起動方法・設定項目・REST APIを解説します。

## 起動方法（3通り）

### (a) コマンドラインから
```bash
deep-research webui                 # http://127.0.0.1:8765
deep-research webui --port 8080     # ポート変更
```

### (b) Pythonモジュールとして
```bash
python -m deep_research_tool.webui.server
```

### (c) このノートブックからバックグラウンド起動（下のセル）


In [ ]:
# ノートブックからバックグラウンドでWeb UIを起動
import threading
from deep_research_tool.webui.server import run_server

HOST, PORT = "127.0.0.1", 8765

server_thread = threading.Thread(
    target=run_server,
    kwargs={"host": HOST, "port": PORT, "output_dir": "./output"},
    daemon=True,   # ノートブック終了時に一緒に止まる
)
server_thread.start()
print(f"Web UI: http://{HOST}:{PORT}")

In [ ]:
# ブラウザを開く（リモートサーバー上ならURLを手元のブラウザに貼り付け）
import webbrowser
webbrowser.open(f"http://{HOST}:{PORT}")

## 画面の設定項目

### 1. 調査テーマ
テーマと要件（含めたい観点）を入力します。

### 情報源（source_mode）
- **Webのみ** / **ローカル文書のみ** / **ハイブリッド**。ローカル文書はサーバーから見えるパスを1行1件で入力

### 収集方法（crawl_mode）
- 標準 / 高速 / AIクロール / **AIクロール（ブラウザ）**
- **ブラウザ**: Chrome / Edge / Firefox
- **WebDriverのパス**: ドライバの自動ダウンロードが失敗する環境（社内プロキシ等）では、
  手動配置したWebDriverのフルパスを指定（例: `C:\tools\msedgedriver.exe`）。
  環境変数 `SELENIUM_DRIVER_PATH` でも可

### 🔌 プロキシ設定
- HTTPS/HTTPプロキシURL、SSL証明書検証のオン/オフ

### 📝 レポートの設定
- 生成エンジン（V1/V2）、文体、出力形式（Markdown/Word/PDF/HTML）、
  チャートライブラリ、図表自動生成、推敲パス

### 🤖 AIモデルの設定
- 既定プロバイダー（OpenAI / Anthropic / ローカルLLM）とモデル
- **APIキー**（空欄=サーバー側の環境変数を使用）
- **APIエンドポイント**: 社内APIゲートウェイ・OpenAI互換サーバー・OllamaのURLを指定
  （空欄=公式API）。プロバイダーに応じて `openai_base_url` / `anthropic_base_url` /
  `local_base_url` として送信されます
- 工程別のAI切り替え（計画/クロール判断/評価/文章作成）

### 2. 進捗と結果
実行中は進捗バーとログが更新され、完了後はレポートと成果物（エビデンス等）を
ダウンロードできます。


## REST API（プログラムからの投稿）

Web UIはシンプルなREST APIで動いており、curlやPythonから直接叩けます。

| メソッド | パス | 説明 |
|---|---|---|
| POST | `/api/research` | 調査ジョブを開始（同時実行は1件） |
| GET | `/api/status` | 現在のジョブの状態・進捗 |
| GET | `/api/reports` | 生成済みレポートの一覧 |
| GET | `/api/report-file?path=...` | レポートファイルの取得（`&download=1`でDL） |


In [ ]:
# POST /api/research — ジョブ投入
import json
from urllib.request import Request, urlopen

BASE = f"http://{HOST}:{PORT}"

params = {
    "query": "洋上風力発電の国内市場動向",
    "requirements": "入札ラウンドの状況と主要プレイヤーを含める",
    "provider": "openai",
    "model": "gpt-5-mini",
    "openai_api_key": "sk-...",                      # 空ならサーバー側の環境変数
    # "openai_base_url": "https://gateway.example.co.jp/v1",  # 社内ゲートウェイ経由なら
    "search_method": "duckduckgo",
    "crawl_mode": "standard",
    # Selenium利用時:
    # "crawl_mode": "ai_crawl_selenium",
    # "browser": "edge",
    # "driver_path": r"C:\tools\msedgedriver.exe",
    "report_version": "v2",
    "v2_writing_style": "business",
    "output_format": "markdown",
    "language": "ja",
}

req = Request(f"{BASE}/api/research",
              data=json.dumps(params).encode(),
              headers={"Content-Type": "application/json"})
with urlopen(req) as r:
    job = json.loads(r.read())
print("job:", job)   # {"job_id": "..."}

In [ ]:
# GET /api/status — 完了までポーリング
import time

while True:
    with urlopen(f"{BASE}/api/status") as r:
        status = json.loads(r.read())
    state = status.get("state")
    print(f"[{status.get('progress', 0):5.1f}%] {state}: {status.get('message', '')}")
    if state in ("completed", "error", "idle"):
        break
    time.sleep(10)

if state == "completed":
    # result には report_path / evidence_json などの成果物パスが入る
    print("成果物:", json.dumps(status.get("result", {}), ensure_ascii=False, indent=2))
elif state == "error":
    print("エラー:", status.get("error"))

In [ ]:
# GET /api/reports — レポート一覧と取得
with urlopen(f"{BASE}/api/reports") as r:
    reports = json.loads(r.read())["reports"]
for f in reports[:5]:
    print(f["name"], f["size"], "bytes")

# 最新レポートの中身を取得
if reports:
    from urllib.parse import quote
    with urlopen(f"{BASE}/api/report-file?path={quote(reports[0]['path'])}") as r:
        content = r.read().decode("utf-8", errors="replace")
    print(content[:500])

## よくあるトラブル

| 症状 | 原因と対処 |
|---|---|
| 「調査を開始」を押しても401/接続エラー | APIキー未設定。フォームのAPIキー欄か、サーバー側の環境変数 `OPENAI_API_KEY` 等を設定 |
| LLM呼び出しがタイムアウト | 社内からは直接公式APIに出られないケース。**APIエンドポイント欄**に社内ゲートウェイURLを設定 |
| Selenium使用時に起動失敗（driver download error） | ドライバの自動DLがプロキシで遮断。**WebDriverのパス欄**に手動配置したドライバを指定 |
| ページ取得が全滅する | プロキシ未設定。🔌プロキシ設定にURLを入力。SSLエラーなら証明書検証をオフ |
| 409 conflict が返る | 既にジョブ実行中（同時実行は1件）。完了を待つ |

## セキュリティ上の注意
- Web UIは認証なしの開発用サーバーです。既定では `127.0.0.1`（ローカルのみ）で
  待ち受けます。`--host 0.0.0.0` で公開する場合は、信頼できるネットワーク内に
  限定してください。
- APIキーをフォームに入れる代わりに、サーバー側の環境変数で設定する方が安全です。
